# Análise Exploratória — Tabelas Gold CVM

Notebook de EDA sobre as agregações da camada Gold do pipeline CVM, no schema `main.cvm_gold.*`. As tabelas são populadas pelo `carregamento_db.py` a partir dos dados limpos em `main.cvm.*`.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Spark já está disponível
print("Setup concluído — pandas, matplotlib, seaborn configurados")

## 1. Visão Geral das Tabelas Gold

In [0]:
tabelas_gold = [
    "main.cvm_gold.patrimonio_por_classe",
    "main.cvm_gold.top_fundos_patrimonio",
    "main.cvm_gold.captacao_resgate_mensal",
    "main.cvm_gold.fundos_ativos",
]

for tabela in tabelas_gold:
    print(f"\n{'='*60}")
    print(f"  {tabela}")
    print(f"{'='*60}")
    df = spark.table(tabela)
    print(f"  Linhas: {df.count():,}")
    print(f"  Colunas: {len(df.columns)}")
    print(f"  Schema:")
    df.printSchema()
    print(f"  Amostra (5 linhas):")
    display(df.limit(5))

## 2. Análise de Patrimônio por Classe de Fundo

In [0]:
df_pc = spark.table("main.cvm_gold.patrimonio_por_classe")

# Última data disponível
ultima_data = df_pc.select("DT_COMPTC").distinct().orderBy("DT_COMPTC", ascending=False).first()[0]
print(f"Última data de competência: {ultima_data}")

# Filtrar última data e top 10 classes por patrimônio
df_ultima = df_pc.filter(f"DT_COMPTC == '{ultima_data}'")
df_pandas = df_ultima.orderBy("vl_patrim_total", ascending=False).limit(10).toPandas()

# Gráfico de barras — top 10 classes por patrimônio
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(
    data=df_pandas,
    y="TP_FUNDO_CLASSE",
    x="vl_patrim_total",
    ax=ax,
    color="#4C72B0",
)
ax.set_title(f"Top 10 Classes por Patrimônio Líquido — {ultima_data}", fontsize=14)
ax.set_xlabel("Patrimônio Líquido Total (R$)")
ax.set_ylabel("Classe")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f} bi"))
plt.tight_layout()
plt.show()

# Tabela detalhada
print(f"\nDetalhes por classe — {ultima_data}:")
display(
    df_ultima.select(
        "TP_FUNDO_CLASSE", "qtd_fundos", "vl_patrim_total",
        "captacao_total", "resgate_total", "fluxo_liquido"
    ).orderBy("vl_patrim_total", ascending=False)
)

## 3. Top 10 Fundos por Patrimônio

In [0]:
df_top = spark.table("main.cvm_gold.top_fundos_patrimonio")

# Tabela com top 10
print("Top 10 Fundos por Patrimônio Líquido:")
display(
    df_top.select(
        "DENOM_SOCIAL", "TP_FUNDO_CLASSE", "VL_PATRIM_LIQ", "VL_QUOTA", "NR_COTST", "ranking"
    ).limit(10)
)

# Gráfico de barras horizontais
df_pandas = df_top.limit(10).toPandas()
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(
    data=df_pandas,
    y="DENOM_SOCIAL",
    x="VL_PATRIM_LIQ",
    ax=ax,
    color="#55A868",
)
ax.set_title("Top 10 Fundos por Patrimônio Líquido", fontsize=14)
ax.set_xlabel("Patrimônio Líquido (R$)")
ax.set_ylabel("Fundo")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f} bi"))
ax.set_yticklabels([(label[:45] + "..." if label and len(label) > 45 else label or "—") for label in df_pandas["DENOM_SOCIAL"]], fontsize=9)
plt.tight_layout()
plt.show()

## 4. Captação e Resgate Mensal

In [0]:
df_cr = spark.table("main.cvm_gold.captacao_resgate_mensal")
df_pandas = df_cr.toPandas()

if len(df_pandas) > 0:
    df_pandas["mes"] = pd.to_datetime(df_pandas["mes"])

    # Gráfico de linha — captação vs resgate
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    ax1 = axes[0]
    ax1.plot(df_pandas["mes"], df_pandas["captacao_total"], marker="o", label="Captação", color="#4C72B0", linewidth=2)
    ax1.plot(df_pandas["mes"], df_pandas["resgate_total"], marker="s", label="Resgate", color="#C44E52", linewidth=2)
    ax1.set_title("Captação e Resgate por Mês", fontsize=14)
    ax1.set_xlabel("Mês")
    ax1.set_ylabel("Valor (R$)")
    ax1.legend()
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f} bi"))
    ax1.tick_params(axis="x", rotation=45)

    # Gráfico de barras — fluxo líquido
    ax2 = axes[1]
    cores = ["#55A868" if v >= 0 else "#C44E52" for v in df_pandas["fluxo_liquido"]]
    ax2.bar(df_pandas["mes"], df_pandas["fluxo_liquido"], color=cores, width=20)
    ax2.set_title("Fluxo Líquido por Mês (Captação - Resgate)", fontsize=14)
    ax2.set_xlabel("Mês")
    ax2.set_ylabel("Fluxo Líquido (R$)")
    ax2.axhline(y=0, color="black", linewidth=0.8, linestyle="--")
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f} bi"))
    ax2.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("Sem dados disponíveis para o gráfico")

# Tabela
print("\nResumo mensal:")
display(df_cr)

## 5. Fundos Ativos

In [0]:
df_fa = spark.table("main.cvm_gold.fundos_ativos")
total_ativos = df_fa.count()
print(f"Total de fundos ativos: {total_ativos}")

# Tabela com fundos ativos
print("\nFundos ativos (amostra):")
display(
    df_fa.select(
        "CNPJ_FUNDO_CLASSE", "DENOM_SOCIAL", "TP_FUNDO_CLASSE", "VL_PATRIM_LIQ"
    ).orderBy("VL_PATRIM_LIQ", ascending=False)
)

# Distribuição por classe
df_classe = df_fa.groupBy("TP_FUNDO_CLASSE").count().orderBy("count", ascending=False).toPandas()

if len(df_classe) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_classe, x="count", y="TP_FUNDO_CLASSE", ax=ax, color="#8172B3")
    ax.set_title("Distribuição de Fundos Ativos por Classe", fontsize=14)
    ax.set_xlabel("Quantidade de Fundos")
    ax.set_ylabel("Classe")
    plt.tight_layout()
    plt.show()
else:
    print("Sem dados de fundos ativos para exibir")